# FarmEasy Mandi Price and Supply Intelligence — Exploratory Data Analysis

This notebook examines the validated output produced by `validate-csv` or `validate-api`. It intentionally does not read raw receipts directly: rejected records and suspicious records are reviewed through the accompanying quality report.

Run it only after loading official source data. The notebook derives findings from the selected dataset rather than hardcoding claims or fabricating prices, arrivals, or supply.

## 1. Setup and input

Set `FARMEASY_EDA_DATA_PATH` to a `validated_mandi_prices.csv` file. Optionally set `FARMEASY_EDA_QUALITY_REPORT_PATH` to its `data_quality_report.json` and `FARMEASY_EDA_COMMODITY` to focus time-series charts.

The expected price unit for the default Data.gov.in resource is INR/quintal. This notebook displays the unit stored in the validated data rather than silently converting it.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', 80)

working_directory = Path.cwd().resolve()
PROJECT_ROOT = next((candidate for candidate in (working_directory, *working_directory.parents) if (candidate / 'analytics').is_dir()), working_directory)
default_data_path = PROJECT_ROOT / 'analytics/data/processed/REPLACE_WITH_RUN_ID/validated_mandi_prices.csv'
DATA_PATH = Path(os.environ.get('FARMEASY_EDA_DATA_PATH', default_data_path))
QUALITY_REPORT_PATH = Path(os.environ['FARMEASY_EDA_QUALITY_REPORT_PATH']) if os.environ.get('FARMEASY_EDA_QUALITY_REPORT_PATH') else None

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Validated dataset not found: {DATA_PATH}. Run validate-csv/validate-api first, then set FARMEASY_EDA_DATA_PATH.'
    )

df = pd.read_csv(DATA_PATH)
required_columns = {
    'market_date', 'state', 'district', 'market', 'commodity',
    'min_price', 'max_price', 'modal_price', 'price_unit',
    'price_spread', 'price_spread_pct', 'quality_status'
}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Validated dataset is missing required columns: {sorted(missing_columns)}')

df['market_date'] = pd.to_datetime(df['market_date'], errors='coerce')
for column in ['min_price', 'max_price', 'modal_price', 'price_spread', 'price_spread_pct']:
    df[column] = pd.to_numeric(df[column], errors='coerce')
for column in ['state', 'district', 'market', 'commodity', 'variety', 'grade', 'quality_status']:
    if column not in df.columns:
        df[column] = 'Unknown'
    df[column] = df[column].fillna('Unknown').astype(str)

price_unit = df['price_unit'].dropna().mode().iat[0] if df['price_unit'].notna().any() else 'stored price unit'
print(f'Loaded {len(df):,} validated observations from {DATA_PATH.name}.')
print(f'Observed date range: {df.market_date.min().date()} to {df.market_date.max().date()} | price unit: {price_unit}')

## 2. Dataset overview and column definitions

The source field `arrival_date` is mapped to `market_date`: it is the date of the published quotation, not proof of an arrival quantity. `modal_price` is the most frequently reported market price, not an arithmetic average.

In [ ]:
column_definitions = pd.DataFrame([
    ('market_date', 'date', 'Official quotation/reporting date'),
    ('state / district / market', 'text', 'Standardized geographic market hierarchy'),
    ('commodity / variety / grade', 'text', 'Product attributes at quotation grain'),
    ('min_price / max_price / modal_price', price_unit, 'Reported daily price range and modal price'),
    ('price_spread / price_spread_pct', price_unit + ' / percent', 'Maximum minus minimum price and relative spread'),
    ('quality_status / quality_warning_codes', 'text', 'Validation outcome; suspicious rows remain visible for review'),
    ('source_observation_hash', 'hash', 'Idempotency and traceability key for the observation grain'),
], columns=['Field', 'Stored type/unit', 'Meaning'])
display(column_definitions)
display(df.head())
df.info()

## 3. Missing values, duplicates, and descriptive statistics

Validated rows should already pass hard price rules. This section checks what remains optional or operationally relevant, rather than assuming cleaning was perfect.

In [ ]:
missing_summary = (df.isna().sum().sort_values(ascending=False).rename('missing_count').to_frame())
missing_summary['missing_pct'] = missing_summary['missing_count'] / len(df) * 100
display(missing_summary[missing_summary['missing_count'] > 0])

duplicate_key = 'source_observation_hash' if 'source_observation_hash' in df.columns else None
duplicate_count = int(df.duplicated(subset=[duplicate_key]).sum()) if duplicate_key else int(df.duplicated().sum())
print(f'Duplicates in validated output: {duplicate_count:,}')
display(df[['min_price', 'max_price', 'modal_price', 'price_spread', 'price_spread_pct']].describe().T)
display(df['quality_status'].value_counts(dropna=False).rename_axis('quality_status').to_frame('observation_count'))

## 4. Price distribution

This histogram shows the distribution of recorded modal prices. A long tail can reflect genuinely different commodities, grades, or markets, so it is not by itself an error signal.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='modal_price', bins=35, kde=True, color='#2E7D32')
plt.title('Distribution of Reported Modal Mandi Prices')
plt.xlabel(f'Modal price ({price_unit})')
plt.ylabel('Number of validated observations')
plt.tight_layout()
plt.show()

## 5. Commodity-level analysis

The bar chart compares the median modal price of the most frequently reported commodities. Median is useful because a few high-price observations do not dominate the comparison.

In [ ]:
commodity_summary = (
    df.groupby('commodity', dropna=False)
      .agg(observation_count=('modal_price', 'size'), median_modal_price=('modal_price', 'median'), average_modal_price=('modal_price', 'mean'))
      .sort_values('observation_count', ascending=False)
)
display(commodity_summary.head(15))
top_commodities = commodity_summary.head(12).reset_index().sort_values('median_modal_price')
plt.figure(figsize=(10, 6))
sns.barplot(data=top_commodities, x='median_modal_price', y='commodity', color='#43A047')
plt.title('Median Modal Price for Most Reported Commodities')
plt.xlabel(f'Median modal price ({price_unit})')
plt.ylabel('Commodity')
plt.tight_layout()
plt.show()

## 6. Market, state, and district comparison

These charts compare the average modal price by geography. They are descriptive: local variety, grade, handling, and reporting coverage can explain a difference.

In [ ]:
state_summary = (df.groupby('state').agg(observation_count=('modal_price', 'size'), average_modal_price=('modal_price', 'mean')).reset_index())
state_summary = state_summary.sort_values('average_modal_price', ascending=False).head(15)
plt.figure(figsize=(10, 6))
sns.barplot(data=state_summary.sort_values('average_modal_price'), x='average_modal_price', y='state', color='#8D6E63')
plt.title('Average Modal Price by State')
plt.xlabel(f'Average modal price ({price_unit})')
plt.ylabel('State')
plt.tight_layout()
plt.show()

market_summary = (
    df.groupby(['state', 'district', 'market'])
      .agg(observation_count=('modal_price', 'size'), average_modal_price=('modal_price', 'mean'), latest_date=('market_date', 'max'))
      .query('observation_count >= 2')
      .sort_values('average_modal_price', ascending=False)
)
display(market_summary.head(20))

## 7. Time-series analysis

This line chart plots daily average modal price and 7/30-day calendar-window rolling averages for one commodity. The selected commodity defaults to the most frequently reported commodity and can be changed with `FARMEASY_EDA_COMMODITY`.

In [ ]:
selected_commodity = os.environ.get('FARMEASY_EDA_COMMODITY') or commodity_summary.index[0]
trend = (
    df.loc[df['commodity'].eq(selected_commodity)]
      .groupby('market_date', as_index=False)['modal_price'].mean()
      .sort_values('market_date')
)
trend['modal_price_7d_ma'] = trend.rolling('7D', on='market_date', min_periods=1)['modal_price'].mean().to_numpy()
trend['modal_price_30d_ma'] = trend.rolling('30D', on='market_date', min_periods=1)['modal_price'].mean().to_numpy()
plt.figure(figsize=(12, 5))
plt.plot(trend['market_date'], trend['modal_price'], alpha=0.45, label='Daily average modal price', color='#6D4C41')
plt.plot(trend['market_date'], trend['modal_price_7d_ma'], label='7-day moving average', color='#2E7D32', linewidth=2)
plt.plot(trend['market_date'], trend['modal_price_30d_ma'], label='30-day moving average', color='#F9A825', linewidth=2)
plt.title(f'Modal Price Trend — {selected_commodity}')
plt.xlabel('Reporting date')
plt.ylabel(f'Average modal price ({price_unit})')
plt.legend()
plt.tight_layout()
plt.show()
display(trend.tail(15))

## 8. Volatility and outlier investigation

Volatility is measured with standard deviation and coefficient of variation (CV). The outlier screen compares an observation with its commodity's 7-day moving average; it is a review aid, not an automatic correction.

In [ ]:
volatility = (
    df.groupby('commodity')
      .agg(observation_count=('modal_price', 'size'), average_modal_price=('modal_price', 'mean'), modal_price_stddev=('modal_price', 'std'))
      .assign(modal_price_cv_pct=lambda x: x['modal_price_stddev'] / x['average_modal_price'] * 100)
      .query('observation_count >= 3')
      .sort_values('modal_price_cv_pct', ascending=False)
)
display(volatility.head(15))
plt.figure(figsize=(10, 6))
sns.barplot(data=volatility.head(12).reset_index(), x='modal_price_cv_pct', y='commodity', color='#EF6C00')
plt.title('Most Volatile Commodities by Modal-Price Coefficient of Variation')
plt.xlabel('Coefficient of variation (%)')
plt.ylabel('Commodity')
plt.tight_layout()
plt.show()

outlier_work = df[['market_date', 'commodity', 'market', 'modal_price', 'quality_status']].copy().sort_values(['commodity', 'market_date'])
outlier_work['commodity_7d_average'] = float('nan')
for _, group in outlier_work.groupby('commodity'):
    ordered_group = group.sort_values('market_date')
    rolling_average = ordered_group.rolling('7D', on='market_date', min_periods=2)['modal_price'].mean().to_numpy()
    outlier_work.loc[ordered_group.index, 'commodity_7d_average'] = rolling_average
outlier_work['deviation_pct'] = (outlier_work['modal_price'] - outlier_work['commodity_7d_average']) / outlier_work['commodity_7d_average'] * 100
outliers = outlier_work.loc[outlier_work['deviation_pct'].abs() >= 50].sort_values('deviation_pct', key=lambda s: s.abs(), ascending=False)
display(outliers.head(25))

## 9. Data-quality conclusions

If a run-level quality report is supplied, this section shows its source counts and issue mix. Otherwise it still summarizes row-level status in the validated dataset. Rejected raw rows are deliberately not reintroduced into price calculations.

In [ ]:
quality_summary = {}
if QUALITY_REPORT_PATH and QUALITY_REPORT_PATH.exists():
    quality_payload = json.loads(QUALITY_REPORT_PATH.read_text(encoding='utf-8'))
    quality_summary = quality_payload.get('summary', {})
    display(pd.DataFrame([quality_summary]))
    display(pd.Series(quality_payload.get('issues_by_code', {}), name='issue_count').sort_values(ascending=False).to_frame())
else:
    display(df['quality_status'].value_counts().rename('validated_observation_count').to_frame())
    print('No quality-report JSON supplied; set FARMEASY_EDA_QUALITY_REPORT_PATH for run-level rejection and gap counts.')

## 10. Generated findings and actionable recommendations

The following cells produce at least five evidence-based findings from the loaded official dataset and three portfolio-ready recommendations. Read the values before quoting them in a report; they change as the source refreshes.

In [ ]:
latest_row = df.loc[df['market_date'].idxmax()]
top_commodity_by_price = commodity_summary['average_modal_price'].idxmax()
most_volatile = volatility.index[0] if not volatility.empty else 'Insufficient observations for a volatility ranking'
best_latest = (
    df.sort_values(['market_date', 'modal_price'], ascending=[False, False])
      .iloc[0]
)
suspicious_count = int(df['quality_status'].eq('suspicious').sum())
stale_count = quality_summary.get('stale_market_count', 'not available without quality report')

findings = [
    f'1. The validated dataset contains {len(df):,} observations from {df.market_date.min().date()} to {df.market_date.max().date()}.',
    f'2. {top_commodity_by_price} has the highest average modal price in this extract ({commodity_summary.loc[top_commodity_by_price].average_modal_price:.2f} {price_unit}).',
    f'3. The highest commodity-level CV is {most_volatile}; interpret it together with its observation count.',
    f'4. The highest latest observed modal price is {best_latest.modal_price:.2f} {price_unit} at {best_latest.market} for {best_latest.commodity} on {best_latest.market_date.date()}.',
    f'5. {suspicious_count:,} validated observations are flagged suspicious; stale-market count is {stale_count}.',
]
for finding in findings:
    display(Markdown(f'- {finding}'))

recommendations = [
    '1. Use the best-mandi price as a shortlist only; subtract transport, handling, commission, quality loss, and buyer-side fees before routing produce.',
    f'2. Monitor {most_volatile} more frequently and avoid treating a one-day move as a stable selling signal.',
    '3. Require a freshness and suspicious-price check before publishing a price comparison to farmers or buyers.',
    '4. Investigate commodities/markets with consistently high price spreads because grade or quality segmentation may improve the interpretation.',
]
for recommendation in recommendations:
    display(Markdown(f'- {recommendation}'))

## Limitations and next steps

- This is descriptive/diagnostic analysis of observed official quotations, not price forecasting.
- The default dataset has no verified arrival quantity or unit, so it must not be used to infer supply.
- Reporting gaps, stale markets, commodity variety, grade, distance, transport cost, and transaction fees can materially change a selling decision.
- A future version can add clearly labeled forecasting only after stable historical coverage and data-quality monitoring are established.